## 2.4 Prediction Engine — Regression (Month-End Discretionary Spend)

Trains a regression model on **all users** to forecast month-end discretionary spend from month-to-date behavior.

From the month-end forecast we derive:
- **Future spending trend** (projected month-end discretionary spend)
- **Overspending risk** (projected month-end > monthly discretionary limit)
- **Budget breach warning** (days-to-limit estimate, linearized v1)

Inputs (read-only):
- `artifacts/transactions_enriched.json` (NDJSON, all users)
- `data/users.csv` (for monthly discretionary limits)

Outputs (artifacts):
- `artifacts/runway_model.pkl` (pickle)
- `artifacts/runway_model_metrics.json`


## Imports

In [3]:
from __future__ import annotations

import pickle
import sys
from datetime import date
from pathlib import Path


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data").is_dir() and (candidate / "artifacts").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate project root containing data/ and artifacts/")


ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from model.predict_core import (
    build_daily_discretionary_spend,
    build_monthly_training_samples,
    default_paths,
    load_user_limits,
    train_month_end_spend_model,
    write_json,
)


## Config

- `CHUNKSIZE` controls NDJSON streaming speed/memory.
- `CUTOFF_DAY` sets the time-based train/test split date (optional).

In [4]:
CHUNKSIZE = 200_000
CUTOFF_DAY: date | None = None  # e.g. date(2017, 12, 31)


## Train model on all users and write artifacts

In [5]:
paths = default_paths()
limits = load_user_limits(paths["users_csv"])

daily = build_daily_discretionary_spend(paths["transactions_ndjson"], chunksize=CHUNKSIZE)
samples = build_monthly_training_samples(daily, user_limits=limits)

artifacts = train_month_end_spend_model(samples, cutoff_day=CUTOFF_DAY)
print("Metrics:", artifacts.metrics)

write_json(paths["metrics_json"], artifacts.metrics)

payload = {
    "feature_columns": list(artifacts.feature_columns),
    "model": artifacts.model,
}
paths["model_pkl"].write_bytes(pickle.dumps(payload))

print("Wrote:", paths["model_pkl"])
print("Wrote:", paths["metrics_json"])


Metrics: {'cutoff_day': '2017-12-15', 'n_samples': 149058, 'n_train': 119283, 'n_test': 29775, 'mae_train_usd': 303.77072986961986, 'mae_test_usd': 285.573654296487}
Wrote: /Users/nicholasp/Personal Coding/JHU/personal finance/artifacts/runway_model.pkl
Wrote: /Users/nicholasp/Personal Coding/JHU/personal finance/artifacts/runway_model_metrics.json
